# DataPulse Local AI — Free Open-Weight LLM Incident Investigator

**Purpose:** Explain real DataPulse synthetic incident reports with a pretrained open-weight LLM **without Gemini API, Vertex AI billing, or any model API key**. Uses [Qwen2.5-1.5B-Instruct](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct) (Apache-2.0). First run downloads model weights from Hugging Face, requiring internet. This is a specialized explanation assistant, **not a newly trained foundation model**, autonomous remediation agent, or proof of production deployment.


## 1. Select runtime and clone the project

In Colab: **Runtime → Change runtime type → T4 GPU** if offered. Free GPU quota is not guaranteed; CPU may be slower. Do not enable a paid subscription. This notebook uses a separate directory and keeps your other DataPulse notebooks unchanged.


In [ ]:
from pathlib import Path
import os, subprocess
repo = Path('/content/aegis-real-time-supply-chain-ai')
if not (repo / '.git').exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/Codewithadi21/aegis-real-time-supply-chain-ai.git',str(repo)],check=True)
else:
    subprocess.run(['git','-C',str(repo),'pull','--ff-only'],check=True)
os.chdir(repo/'datapulse-ai')
print('Working directory:',os.getcwd())


## 2. Install dependencies and verify the core project

The LLM weights are not downloaded by installing Python libraries. Download occurs when the model is first loaded.


In [ ]:
%pip -q install -r requirements-local-llm.txt
!python -m pytest -q


## 3. Generate DataPulse's real structured incident report

This is the **same tested DataPulse replay and deterministic investigator**, not fabricated LLM evidence.


In [ ]:
from datapulse.replay import decision_replay
from datapulse.investigator import investigate
from datapulse.local_llm import build_incident_messages, explain_deterministic
import json
summary, comparisons = decision_replay()
report = investigate(summary)
print(json.dumps(report, indent=2))
assert 'incident_type' in report and 'evidence' in report
print('Validated report ready for local LLM')


## 4. Show the offline fallback (always works without downloading a model)

This is a deterministic, transparently labeled explanation, **not** LLM output.


In [ ]:
print(explain_deterministic(report))


## 5. Download and load the open-weight model

First run needs public internet access to Hugging Face. `Qwen2.5-1.5B-Instruct` is a real instruction-tuned model, not Gemini and not trained by DataPulse. It may use several GB of RAM/VRAM. If Colab has no GPU, expect slower generation. An out-of-memory error means you may need a smaller model such as `Qwen/Qwen2.5-0.5B-Instruct`.


In [ ]:
import torch
from datapulse.local_llm import load_local_model, explain_local
MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'
print('GPU available:',torch.cuda.is_available())
tokenizer, local_model = load_local_model(MODEL_ID)
print('Model loaded:',MODEL_ID)


## 6. Let the local AI explain the actual DataPulse incident

The underlying measurements and incident rules still come from Python, not the LLM. The LLM is constrained by the supplied JSON evidence, but its output should be reviewed for unsupported statements.


In [ ]:
explanation = explain_local(report, tokenizer, local_model, max_new_tokens=320)
print('DATAPULSE LOCAL AI — GENERATED INCIDENT EXPLANATION\n')
print(explanation)


## 7. Save portfolio evidence locally

The generation is an experiment. Saving the exact grounded report and explanation helps reviewers reproduce it. Do not commit private company data or token files.


In [ ]:
from pathlib import Path
p = Path('artifacts'); p.mkdir(exist_ok=True)
(p/'local_llm_incident_report.json').write_text(json.dumps(report,indent=2),encoding='utf-8')
(p/'local_llm_explanation.txt').write_text(explanation,encoding='utf-8')
print('Saved:',p/'local_llm_incident_report.json',p/'local_llm_explanation.txt')


## Troubleshooting and limitations

- **No Google API key needed:** do not run `google.genai` or Vertex AI in this notebook
- **Hugging Face download error:** retry when Colab internet is available, or check Hugging Face access
- **CUDA out of memory:** use `MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'`, restart runtime, rerun
- **CPU runtime:** model can run much more slowly; T4 is recommended when available
- **Model quality:** small open-weight model can hallucinate, omit nuance, or misunderstand metrics; verify everything against JSON
- **No autonomous tools or RAG:** this version demonstrates an LLM-backed explanation adapter; BigQuery tool retrieval or RAG can be added later

**Architecture:** DataPulse deterministic replay → `investigate(summary)` → trusted report → `build_incident_messages(report)` → local Qwen model → human-reviewed explanation. The model is not trained or fine-tuned by this notebook.
